# Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import warnings
from sklearn.preprocessing import OrdinalEncoder
warnings.filterwarnings('ignore')
print(f'imports successful')

imports successful


In [2]:
# Feature whitelist from data preparation / EDA
with open('../data/processed/consistent_features.json') as f:
    cfg = json.load(f)
CONSISTENT_FEATURES = cfg['consistent_features']
MODEL_EXCLUDED      = cfg['model_excluded']     # grade, sub_grade, int_rate
LEAKAGE_COLUMNS     = cfg['leakage_columns']

print(f'successful')

successful


## Feature Engineering Function — Affordability, Credit Quality, Debt Burden, Employment & Vintage

One function, applied identically to the development and production samples. All bins use `right=False` (left-inclusive). The three credit-quality dimensions (DTI, FICO, revolving utilisation) are kept **continuous** so the models can choose their own split points rather than inherit fixed buckets as this improved the model performance during the trial phase; `open_acc` remains an ordinal band.

In [3]:
# Defined one function to serve both our 100k dev sample and 500k prod samples
# The features were arranged based on common attributes (Data-quality fixes,Affordability & credit quality,Employment & credit behaviour
# Loan purpose risk tier)

def engineer(df):
    """Origination-time feature engineering — single source of truth for dev and production."""
    df = df.copy()

    # Vintage identifiers from the origination date (structural time, not a macro covariate)
    issue = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
    df['vintage_year']    = issue.dt.year
    df['vintage_quarter'] = issue.dt.quarter

    # Data-quality fixes
    df['annual_inc']     = df['annual_inc'].clip(upper=df['annual_inc'].quantile(0.99))   # cap extreme incomes
    df['log_annual_inc'] = np.log1p(df['annual_inc'])                                      # compress income skew
    df['revol_util']     = df['revol_util'].fillna(df['revol_util'].median())             # impute missing utilisation
    df['dti']            = df['dti'].clip(upper=df['dti'].quantile(0.99)).fillna(df['dti'].median())  # winsorise 999 sentinel + impute

    # Affordability & credit quality (continuous) 
    df['loan_to_income'] = df['loan_amnt'] / df['annual_inc'].clip(lower=1)   # higher = more stretched borrower
    df['fico_orig']      = (df['fico_range_low'] + df['fico_range_high']) / 2  # origination FICO midpoint

    # Employment & credit behaviour 
    df['employment_years'] = (df['emp_length'].replace({'< 1 year': '0.5 years'})
                                .str.extract(r'(\d+\.?\d*)').astype(float).fillna(0))
    df['short_tenure_flag'] = (df['employment_years'] < 1).astype(int)        # thin tenure (<1yr) instability marker
    df['open_acc_band']     = pd.cut(df['open_acc'], [0, 5, 10, 20, np.inf],
                                     labels=['Few', 'Some', 'Many', 'VeryMany'], right=False)
    df['delinq_flag']       = (df['delinq_2yrs'] > 0).astype(int)             # any delinquency in last 2 yrs
    df['pub_rec_flag']      = (df['pub_rec'] > 0).astype(int)                 # any derogatory public record

    # Loan purpose risk tier (from observed EDA default rates; Nan(missing values) -> Med)
    # purpose_risk_tier — purposes grouped by observed default rate from EDA
    purpose_tiers = {
    'Low':  ['car', 'wedding', 'home_improvement', 'credit_card', 'educational'],   # < 19%   
    'Med':  ['major_purchase', 'vacation', 'other', 'debt_consolidation'],          # 19%–22% 
    'High': ['medical', 'house', 'moving', 'renewable_energy', 'small_business'],    # >= 22%  
    }
    # Flatten to {purpose: tier} for the mapping
    purpose_map = {p: tier for tier, purposes in purpose_tiers.items() for p in purposes}

    df['purpose_risk_tier'] = df['purpose'].map(purpose_map).fillna('Med')
    return df

## Apply to the development sample (100k)

In [4]:
# Call engineer function on 100k dev sample
df = engineer(pd.read_parquet(f'../data/processed/loans_dev_100k.parquet'))
print(f'Engineered dev sample: {df.shape}')

Engineered dev sample: (100000, 164)


### Build the model feature list
The three credit-quality measures enter as their raw continuous values — dti, revol_util, and fico_orig (the midpoint of the FICO range) — rather than the coarse bands used earlier, which discarded signal and lowered model performance, and income enters as log_annual_inc to temper its skew. Raw columns such as loan_amnt and home_ownership are listed directly alongside the engineered features rather than routed through a separate passthrough audit, because the only leakage control that matters is the deliberate exclusion of LendingClub's own risk-pricing fields (grade, sub_grade, int_rate), which are simply never added to the list. A final existence check retains only the columns actually present in the dataframe, guarding against a mistyped feature name.The final feature set is a single curated list of fifteen origination-time predictors. 

In [5]:
FEATURE_LIST = [
    'loan_to_income', 'dti', 'fico_orig', 'revol_util',
    'employment_years', 'short_tenure_flag', 'vintage_year', 'vintage_quarter',
    'open_acc_band', 'delinq_flag', 'pub_rec_flag', 'purpose_risk_tier',
    'loan_amnt', 'log_annual_inc', 'home_ownership',
]
# Safety: keep only columns actually engineered into df
FEATURE_LIST = [f for f in FEATURE_LIST if f in df.columns]

print(f'Final feature count: {len(FEATURE_LIST)}')
print(FEATURE_LIST)

Final feature count: 15
['loan_to_income', 'dti', 'fico_orig', 'revol_util', 'employment_years', 'short_tenure_flag', 'vintage_year', 'vintage_quarter', 'open_acc_band', 'delinq_flag', 'pub_rec_flag', 'purpose_risk_tier', 'loan_amnt', 'log_annual_inc', 'home_ownership']


### Vintage Cohort Train, Validation and Test Split

Temporal split by `vintage_year` — train on earlier cohorts, validate and test on progressively later ones to simulate out-of-time production scoring and prevent vintage-level leakage from inflating evaluation metrics.

In [6]:
# Temporal split: train on vintages up to 2015, validate on 2016, and test on 2017–2018
train_mask = df['vintage_year'] <= 2015
val_mask   = df['vintage_year'] == 2016
test_mask  = df['vintage_year'].isin([2017, 2018])

# Create feature (X) and target (y) datasets for each temporal split
X_train = df.loc[train_mask, FEATURE_LIST].copy()
y_train = df.loc[train_mask, 'default_flag'].copy()

X_val = df.loc[val_mask, FEATURE_LIST].copy()
y_val = df.loc[val_mask, 'default_flag'].copy()

X_test = df.loc[test_mask, FEATURE_LIST].copy()
y_test = df.loc[test_mask, 'default_flag'].copy()

# Check sample size and default rate in each split
for name, (X, y) in {
    'Train': (X_train, y_train),
    'Validation': (X_val, y_val),
    'Test': (X_test, y_test)
}.items():
    print(f'{name}: {len(X):,} rows | Default Rate: {y.mean():.2%}')

Train: 61,442 rows | Default Rate: 18.43%
Validation: 21,786 rows | Default Rate: 23.29%
Test: 16,772 rows | Default Rate: 21.29%


### Train / Validation / Test Split — Temporal Approach

**Why use a temporal split?**  
Loans are partitioned by origination year rather than randomly to better reflect real-world deployment. This ensures models are trained on historical loans and evaluated on future vintages, reducing the risk of information leakage and providing a more realistic assessment of performance.

**Why these periods?**  
Loans originated up to 2015 are used for training to maximize development data. The 2016 vintage serves as the validation set for model tuning and selection, while 2017–2018 loans form the out-of-time test set. Combining 2017 and 2018 provides a sufficiently mature test population and avoids the instability that would result from relying on 2018 alone.

**Why is validation harder?**  
The 2016 vintage exhibits a higher default rate than the training period, reflecting a known deterioration in credit quality during that year. As a result, validation performance is assessed under more challenging conditions, providing a robust test of model generalization.

### Encode Categorical Variables and Save Modeling Datasets

Only three categoricals require encoding — `open_acc_band`, `purpose_risk_tier`, and `home_ownership`. The encoder is fit on **train only** and applied to validation and test without refitting.

In [7]:
# Review data types of all model features
df[FEATURE_LIST].dtypes

loan_to_income        float64
dti                   float64
fico_orig             float64
revol_util            float64
employment_years      float64
short_tenure_flag       int64
vintage_year            int32
vintage_quarter         int32
open_acc_band        category
delinq_flag             int64
pub_rec_flag            int64
purpose_risk_tier      object
loan_amnt             float64
log_annual_inc        float64
home_ownership         object
dtype: object

In [8]:
# Three remaining categoricals (vintage_year/quarter are already int; the rest are continuous)
CAT_COLS = ['open_acc_band', 'purpose_risk_tier', 'home_ownership']

encoder = OrdinalEncoder(
    categories=[
        ['Few', 'Some', 'Many', 'VeryMany'],                  # open_acc_band
        ['Low', 'Med', 'High'],                               # purpose_risk_tier
        ['NONE', 'ANY', 'OTHER', 'RENT', 'OWN', 'MORTGAGE'],  # home_ownership
    ],
    handle_unknown='use_encoded_value', unknown_value=-1
)

X_train_enc, X_val_enc, X_test_enc = X_train.copy(), X_val.copy(), X_test.copy()
X_train_enc[CAT_COLS] = encoder.fit_transform(X_train[CAT_COLS].astype(str))
X_val_enc[CAT_COLS]   = encoder.transform(X_val[CAT_COLS].astype(str))
X_test_enc[CAT_COLS]  = encoder.transform(X_test[CAT_COLS].astype(str))

print(f'successful')

successful


In [9]:
# Save modelling artifacts: encoder, feature list, and six split parquets of train/validation/test datasets

# Save encoded feature matrices
joblib.dump(encoder, f'../models/ordinal_encoder.pkl')

# Save feature list used for modelling
with open(f'../models/feature_list.txt', 'w') as f:
    f.write('\n'.join(FEATURE_LIST))

# Save train/validation/test datasets
for name, X in [('X_train', X_train_enc), ('X_val', X_val_enc), ('X_test', X_test_enc)]:
    X.to_parquet(f'../data/processed/{name}.parquet', index=False)
for name, y in [('y_train', y_train), ('y_val', y_val), ('y_test', y_test)]:
    y.to_frame().to_parquet(f'../data/processed/{name}.parquet', index=False)

print('Saved: encoder, feature_list.txt, and all 6 split parquets')
print()
print(f'Total number of features {len(FEATURE_LIST)}')
print(f'\nFeature List:\n{FEATURE_LIST}')
print()
print(f'\nMissing items:\n{df[FEATURE_LIST].isna().sum()}')

Saved: encoder, feature_list.txt, and all 6 split parquets

Total number of features 15

Feature List:
['loan_to_income', 'dti', 'fico_orig', 'revol_util', 'employment_years', 'short_tenure_flag', 'vintage_year', 'vintage_quarter', 'open_acc_band', 'delinq_flag', 'pub_rec_flag', 'purpose_risk_tier', 'loan_amnt', 'log_annual_inc', 'home_ownership']


Missing items:
loan_to_income       0
dti                  0
fico_orig            0
revol_util           0
employment_years     0
short_tenure_flag    0
vintage_year         0
vintage_quarter      0
open_acc_band        0
delinq_flag          0
pub_rec_flag         0
purpose_risk_tier    0
loan_amnt            0
log_annual_inc       0
home_ownership       0
dtype: int64


## Apply to the Engineer function to the 500k Production Sample

The 500k production sample is run through the **same `engineer()` function** and the **same fitted encoder** (transform only, never refit), then saved. This lets `04_modeling` retrain the champion and run the temporal-drift analysis by simply loading one file — no feature logic is repeated outside this notebook.

In [10]:
# Load and engineer production data from defined function
prod = engineer(pd.read_parquet(f'../data/processed/loans_prod_500k.parquet'))

# Apply training encoder to categorical features
prod[CAT_COLS] = encoder.transform(prod[CAT_COLS].astype(str))
print('Encoded production categoricals')

successful


### Save Modeling Datasets

In [11]:
# Save engineered production dataset
prod[FEATURE_LIST + ['default_flag']].to_parquet(
    f'../data/processed/loans_prod_engineered.parquet',
    index=False
)
print(f'Saved: loans_prod_engineered.parquet | {prod[FEATURE_LIST].shape[0]:,} rows x {len(FEATURE_LIST)} features')
print()
print(f"Vintage spread: \n{prod['vintage_year'].value_counts().sort_index()}")

Saved: loans_prod_engineered.parquet | 500,000 rows x 15 features

Vintage spread: 
vintage_year
2007        93
2008       581
2009      1753
2010      4288
2011      8072
2012     19833
2013     50100
2014     82916
2015    139572
2016    108933
2017     62928
2018     20931
Name: count, dtype: int64


In [19]:
# Preview a random sample of engineered production features
pd.read_parquet('../data/processed/loans_prod_engineered.parquet')[FEATURE_LIST].sample(5).head()

,loan_to_income,dti,fico_orig,revol_util,employment_years,short_tenure_flag,vintage_year,vintage_quarter,open_acc_band,delinq_flag,pub_rec_flag,purpose_risk_tier,loan_amnt,log_annual_inc,home_ownership
4262,0.097674,8.62,717.0,95.6,10.0,0,2010,4,1.0,0,0,2.0,4200.0,10.668979,5.0
328540,0.236842,15.47,672.0,56.5,7.0,0,2016,3,2.0,0,0,1.0,7200.0,10.322231,3.0
441264,0.373333,2.74,757.0,35.6,0.5,1,2017,3,0.0,0,0,0.0,28000.0,11.225257,5.0
272747,0.237838,12.29,707.0,38.4,4.0,0,2015,2,1.0,0,0,1.0,22000.0,11.434975,5.0
381570,0.500000,29.56,732.0,33.0,5.0,0,2016,4,1.0,0,0,1.0,15000.0,10.308986,3.0
